In [ ]:
import os
import sys

ENDWITHS = 'Pipelines'

NOTEBOOK_DIR = os.getcwd()

if not NOTEBOOK_DIR.endswith(ENDWITHS):
    raise ValueError(f"Not in correct dir, expect end with {ENDWITHS}, but got {NOTEBOOK_DIR} instead")

BASE_DIR = os.path.join(NOTEBOOK_DIR, '..', '..','..')

sys.path.insert(0, BASE_DIR)

In [ ]:
from src.pipeline.SegmentationModels.YoloSeg import YoloSeg, plot_image
import cv2
from huggingface_hub import hf_hub_download

frame_seg_path = hf_hub_download(
    repo_id = "TheBlindMaster/yolov8n-manga-frame-seg",
    filename = "best.pt"
)

bubble_seg_path = hf_hub_download(
    repo_id = "TheBlindMaster/yolov8n-manga-bubble-seg",
    filename = "best.pt"
)

EX_IMG_PATH = os.path.join(BASE_DIR, "data/Manga109_released_2023_12_07/images/AisazuNihaIrarenai/007.jpg")

In [ ]:
device = "mps"

panel_detector = YoloSeg(model_pt_path=frame_seg_path, device=device, verbose=True)

bubble_detector = YoloSeg(model_pt_path=bubble_seg_path, device=device, verbose=True)

panel_detector.load_model()
bubble_detector.load_model()

image = cv2.imread(EX_IMG_PATH)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# image, bboxes, masks, _ = panel_detector.predict(image, plot=True, plot_bbox=True)


```
merge happend = 1

def merg(box 1, mask 1, box 2, mask 2):
    box new = box 1 merge box 2 --> polygon
    mask new = mask 2 merge mask 2 --> polyon
    return box new, mask new

list bboxes, list mask
        

group by row, let say max row for each page is 4 (fucking arbitrary choice), row height = image heigh mod 4
calculate center

list bboxes, list masks, list centers

calculate row group by center mod row height

list bboxes, list masks, list centers, list row group

def search merge( list bboxes, list masks):
    for i, box1 in enumerate(bboxes):
        for j, box2 in enumerate(bboxes[i+1:])
            if (frame 1 intersect frame 2) > min area( frame 1, frame 2) / 5 (5 is also an arbitrary choice)
                merge happend = 1
                return merge happend, i, j
    return 0, 0, 0

everything group by row group;
    merge happend = 1
    while merge happend = 1:
        merge happend, i, j = search merge (list bboxes, list masks)
        if merge happend = 1
            bboxes, maskes = do merge (i, j, bboxes, masks)

now we have cleaned / merged panel

ordering panel 

bboxes, masks, grouped by row group
    sort bboxes, masks key = x cordinate of center

concatnate all row group by row group --> order list of bboxes and masks of the frame

map bubble to corresponding frame

for bubble in list b

```

In [ ]:
import numpy as np
import cv2
from typing import List, Tuple, Dict, Any, Optional
from shapely.geometry import box

# === HELPER FUNCTIONS ===

def calculate_center(bbox: List[float]) -> Tuple[float, float]:
    x1, y1, x2, y2 = bbox
    return ((x1 + x2) / 2, (y1 + y2) / 2)

def get_row_group(center_y: float, row_height: float) -> int:
    return int(center_y // row_height)

def calculate_intersection_ratio(bbox1: List[float], bbox2: List[float]) -> float:
    box1 = box(*bbox1)
    box2 = box(*bbox2)
    
    if not box1.intersects(box2):
        return 0.0
    
    intersection_area = box1.intersection(box2).area
    min_area = min(box1.area, box2.area)
    
    return intersection_area / min_area if min_area > 0 else 0.0

def merge_bboxes(bbox1: List[float], bbox2: List[float]) -> List[float]:
    x1 = min(bbox1[0], bbox2[0])
    y1 = min(bbox1[1], bbox2[1])
    x2 = max(bbox1[2], bbox2[2])
    y2 = max(bbox1[3], bbox2[3])
    return [x1, y1, x2, y2]

def merge_masks(mask1: np.ndarray, mask2: np.ndarray) -> np.ndarray:
    return np.logical_or(mask1, mask2).astype(mask1.dtype)

def search_merge(bboxes: List[List[float]], threshold: float = 0.2) -> Tuple[bool, int, int]:
    for i in range(len(bboxes)):
        for j in range(i + 1, len(bboxes)):
            ratio = calculate_intersection_ratio(bboxes[i], bboxes[j])
            if ratio > threshold:
                return True, i, j
    return False, -1, -1

def merge_overlapping_panels(
    bboxes: List[List[float]], 
    masks: List[np.ndarray],
    threshold: float = 0.2
) -> Tuple[List[List[float]], List[np.ndarray]]:
    bboxes = list(bboxes)
    masks = list(masks)
    
    merge_happened = True
    while merge_happened:
        merge_happened, i, j = search_merge(bboxes, threshold)
        if merge_happened:
            new_bbox = merge_bboxes(bboxes[i], bboxes[j])
            new_mask = merge_masks(masks[i], masks[j])
            
            bboxes.pop(j)
            bboxes.pop(i)
            masks.pop(j)
            masks.pop(i)
            
            bboxes.append(new_bbox)
            masks.append(new_mask)
    
    return bboxes, masks

def group_by_row(
    bboxes: List[List[float]], 
    masks: List[np.ndarray],
    image_height: int,
    num_rows: int = 4
) -> Dict[int, List[Tuple[List[float], np.ndarray, float]]]:
    row_height = image_height / num_rows
    groups = {}
    
    for bbox, mask in zip(bboxes, masks):
        center_x, center_y = calculate_center(bbox)
        row_group = get_row_group(center_y, row_height)
        
        if row_group not in groups:
            groups[row_group] = []
        groups[row_group].append((bbox, mask, center_x))
    
    return groups

def sort_panels_manga_order(
    bboxes: List[List[float]], 
    masks: List[np.ndarray],
    image_height: int,
    num_rows: int = 4
) -> Tuple[List[List[float]], List[np.ndarray]]:
    if len(bboxes) == 0:
        return [], []
    
    bboxes, masks = merge_overlapping_panels(bboxes, masks)
    row_groups = group_by_row(bboxes, masks, image_height, num_rows)
    
    sorted_bboxes = []
    sorted_masks = []
    
    for row_idx in sorted(row_groups.keys()):
        row_items = row_groups[row_idx]
        row_items.sort(key=lambda x: x[2], reverse=True)
        
        for bbox, mask, _ in row_items:
            sorted_bboxes.append(bbox)
            sorted_masks.append(mask)
    
    return sorted_bboxes, sorted_masks

def sort_bubbles_in_panel_by_rows(
    bubbles: List[Dict[str, Any]],
    num_rows: int = 3,
    right_to_left: bool = True
) -> List[Dict[str, Any]]:
    """
    Sort bubbles within a panel using row-based grouping.
    Same logic as panel sorting but applied to bubbles.
    """
    if len(bubbles) == 0:
        return []
    
    if len(bubbles) == 1:
        return bubbles
    
    # Calculate panel bounds from bubbles
    all_y_min = min(b['ymin'] for b in bubbles)
    all_y_max = max(b['ymax'] for b in bubbles)
    panel_height = all_y_max - all_y_min
    
    if panel_height <= 0:
        return bubbles
    
    row_height = panel_height / num_rows
    
    # Group bubbles by row
    row_groups = {}
    for bubble in bubbles:
        center_y = (bubble['ymin'] + bubble['ymax']) / 2
        center_x = (bubble['xmin'] + bubble['xmax']) / 2
        
        # Calculate row relative to panel bounds
        relative_y = center_y - all_y_min
        row_idx = int(relative_y // row_height)
        
        if row_idx not in row_groups:
            row_groups[row_idx] = []
        row_groups[row_idx].append((bubble, center_x))
    
    # Sort each row and concatenate
    sorted_bubbles = []
    for row_idx in sorted(row_groups.keys()):  # Top to bottom
        row_items = row_groups[row_idx]
        # Sort by x: descending for RTL manga, ascending for LTR
        row_items.sort(key=lambda x: x[1], reverse=right_to_left)
        sorted_bubbles.extend([item[0] for item in row_items])
    
    return sorted_bubbles


def sort_bubbles_in_panel_adaptive(
    bubbles: List[Dict[str, Any]],
    right_to_left: bool = True
) -> List[Dict[str, Any]]:
    """
    Adaptively determine number of rows based on bubble positions.
    Better for panels with varying layouts.
    """
    if len(bubbles) <= 1:
        return bubbles
    
    # Get all bubble centers
    centers = []
    for b in bubbles:
        cy = (b['ymin'] + b['ymax']) / 2
        cx = (b['xmin'] + b['xmax']) / 2
        centers.append((b, cx, cy))
    
    # Sort by y to find natural row breaks
    centers_sorted_y = sorted(centers, key=lambda x: x[2])
    
    # Detect row breaks using y-gap threshold
    avg_bubble_height = np.mean([b['ymax'] - b['ymin'] for b in bubbles])
    gap_threshold = avg_bubble_height * 0.5  # Bubbles in same row if gap < 50% of height
    
    rows = []
    current_row = [centers_sorted_y[0]]
    
    for i in range(1, len(centers_sorted_y)):
        prev_y = centers_sorted_y[i - 1][2]
        curr_y = centers_sorted_y[i][2]
        
        if curr_y - prev_y > gap_threshold:
            # New row
            rows.append(current_row)
            current_row = [centers_sorted_y[i]]
        else:
            current_row.append(centers_sorted_y[i])
    
    rows.append(current_row)  # Don't forget last row
    
    # Sort each row by x and concatenate
    sorted_bubbles = []
    for row in rows:
        row.sort(key=lambda x: x[1], reverse=right_to_left)
        sorted_bubbles.extend([item[0] for item in row])
    
    return sorted_bubbles

def map_bubbles_to_panels(
    bubbles: List[Dict[str, Any]],
    panel_bboxes: List[List[float]]
) -> Dict[int, List[Dict[str, Any]]]:
    panel_bubbles = {i: [] for i in range(len(panel_bboxes))}
    unassigned = []
    
    for bubble in bubbles:
        bubble_center = ((bubble['xmin'] + bubble['xmax']) / 2, 
                         (bubble['ymin'] + bubble['ymax']) / 2)
        assigned = False
        
        for i, panel_bbox in enumerate(panel_bboxes):
            x1, y1, x2, y2 = panel_bbox
            if x1 <= bubble_center[0] <= x2 and y1 <= bubble_center[1] <= y2:
                panel_bubbles[i].append(bubble)
                assigned = True
                break
        
        if not assigned:
            unassigned.append(bubble)
    
    if unassigned:
        panel_bubbles[-1] = unassigned
    
    return panel_bubbles

def get_ordered_bubbles(
    bubbles: List[Dict[str, Any]],
    panel_bboxes: List[List[float]],
    panel_masks: List[np.ndarray],
    image_height: int,
    num_rows: int = 3,
    bubble_row_method: str = 'adaptive'  # 'adaptive' or 'fixed'
) -> List[Dict[str, Any]]:
    if len(bubbles) == 0:
        return []
    
    sorted_panels, sorted_masks = sort_panels_manga_order(
        panel_bboxes, panel_masks, image_height, num_rows
    )
    
    panel_bubbles = map_bubbles_to_panels(bubbles, sorted_panels)
    
    ordered_bubbles = []
    for i in range(len(sorted_panels)):
        panel_b = panel_bubbles.get(i, [])
        if bubble_row_method == 'adaptive':
            sorted_panel_b = sort_bubbles_in_panel_adaptive(panel_b, right_to_left=True)
        else:
            sorted_panel_b = sort_bubbles_in_panel_by_rows(panel_b, num_rows=3, right_to_left=True)
        ordered_bubbles.extend(sorted_panel_b)
    
    if -1 in panel_bubbles:
        unassigned = sort_bubbles_in_panel_adaptive(panel_bubbles[-1], right_to_left=True)
        ordered_bubbles.extend(unassigned)
    
    return ordered_bubbles

# === DOUBLE PAGE HANDLING ===

def is_double_page(image: np.ndarray) -> bool:
    height, width = image.shape[:2]
    return width / height > 1.0

def split_double_page(image: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    height, width = image.shape[:2]
    mid = width // 2
    right_page = image[:, mid:]
    left_page = image[:, :mid]
    return right_page, left_page

def adjust_mask_for_right_page(mask: np.ndarray, full_width: int) -> np.ndarray:
    height = mask.shape[0]
    page_width = mask.shape[1]
    full_mask = np.zeros((height, full_width), dtype=mask.dtype)
    mid = full_width // 2
    full_mask[:, mid:mid + page_width] = mask
    return full_mask

def adjust_mask_for_left_page(mask: np.ndarray, full_width: int) -> np.ndarray:
    height = mask.shape[0]
    page_width = mask.shape[1]
    full_mask = np.zeros((height, full_width), dtype=mask.dtype)
    full_mask[:, :page_width] = mask
    return full_mask

def process_single_page(
    image: np.ndarray,
    detect_panels_fn,
    detect_bubbles_fn,
    num_rows: int = 4
) -> Tuple[List[List[float]], List[np.ndarray], List[Dict[str, Any]]]:
    panel_bboxes, panel_masks = detect_panels_fn(image)
    bubbles = detect_bubbles_fn(image)
    
    image_height = image.shape[0]
    
    sorted_panels, sorted_masks = sort_panels_manga_order(
        panel_bboxes, panel_masks, image_height, num_rows
    )
    
    ordered_bubbles = get_ordered_bubbles(
        bubbles, panel_bboxes, panel_masks, image_height, num_rows
    )
    
    return sorted_panels, sorted_masks, ordered_bubbles

def process_manga_page(
    image: np.ndarray,
    detect_panels_fn,
    detect_bubbles_fn,
    num_rows: int = 4
) -> Tuple[List[List[float]], List[np.ndarray], List[Dict[str, Any]]]:
    if not is_double_page(image):
        return process_single_page(image, detect_panels_fn, detect_bubbles_fn, num_rows)
    
    height, full_width = image.shape[:2]
    mid_x = full_width // 2
    
    right_page, left_page = split_double_page(image)
    
    right_panels, right_masks, right_bubbles = process_single_page(
        right_page, detect_panels_fn, detect_bubbles_fn, num_rows
    )
    
    left_panels, left_masks, left_bubbles = process_single_page(
        left_page, detect_panels_fn, detect_bubbles_fn, num_rows
    )
    
    adjusted_right_panels = [
        [x1 + mid_x, y1, x2 + mid_x, y2] 
        for x1, y1, x2, y2 in right_panels
    ]
    adjusted_right_masks = [
        adjust_mask_for_right_page(mask, full_width) 
        for mask in right_masks
    ]
    adjusted_right_bubbles = [
        {**b, 'xmin': b['xmin'] + mid_x, 'xmax': b['xmax'] + mid_x}
        for b in right_bubbles
    ]
    
    adjusted_left_masks = [
        adjust_mask_for_left_page(mask, full_width) 
        for mask in left_masks
    ]
    
    all_panels = adjusted_right_panels + left_panels
    all_masks = adjusted_right_masks + adjusted_left_masks
    all_bubbles = adjusted_right_bubbles + left_bubbles
    
    return all_panels, all_masks, all_bubbles

In [ ]:
import numpy as np
from typing import List, Dict, Any, Tuple, Optional
from PIL import Image

def is_double_page(image: np.ndarray) -> bool:
    """Check if image is a double-page spread (width > height)"""
    height, width = image.shape[:2]
    return width / height > 1.0

def split_double_page(image: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Split double-page into right and left halves (manga order)"""
    height, width = image.shape[:2]
    mid = width // 2
    
    right_page = image[:, mid:]  # Right page first for manga
    left_page = image[:, :mid]
    
    return right_page, left_page

def adjust_bbox_for_left_page(bbox: List[float], page_width: int) -> List[float]:
    """Adjust bbox coordinates when merging left page back"""
    x1, y1, x2, y2 = bbox
    return [x1 + page_width, y1, x2 + page_width, y2]

def adjust_mask_for_left_page(mask: np.ndarray, page_width: int, full_width: int) -> np.ndarray:
    """Pad mask to fit in left side of full image"""
    height = mask.shape[0]
    full_mask = np.zeros((height, full_width), dtype=mask.dtype)
    full_mask[:, page_width:page_width + mask.shape[1]] = mask
    return full_mask

def adjust_mask_for_right_page(mask: np.ndarray, full_width: int) -> np.ndarray:
    """Pad mask to fit in right side of full image"""
    height = mask.shape[0]
    page_width = mask.shape[1]
    full_mask = np.zeros((height, full_width), dtype=mask.dtype)
    # Right page goes in the right half (starts at mid point)
    mid = full_width // 2
    full_mask[:, mid:mid + page_width] = mask
    return full_mask

def adjust_bubble_for_left_page(bubble: Dict[str, Any], page_width: int) -> Dict[str, Any]:
    """Adjust bubble coordinates when merging left page back"""
    return {
        **bubble,
        'xmin': bubble['xmin'] + page_width,
        'xmax': bubble['xmax'] + page_width
    }

def adjust_bubble_for_right_page(bubble: Dict[str, Any], mid_x: int) -> Dict[str, Any]:
    """Adjust bubble coordinates for right page (which starts at mid_x)"""
    return {
        **bubble,
        'xmin': bubble['xmin'] + mid_x,
        'xmax': bubble['xmax'] + mid_x
    }

def process_single_page(
    image: np.ndarray,
    detect_panels_fn,
    detect_bubbles_fn,
    num_rows: int = 4
) -> Tuple[List[List[float]], List[np.ndarray], List[Dict[str, Any]]]:
    """Process a single page - detect panels and bubbles, return sorted"""
    panel_bboxes, panel_masks = detect_panels_fn(image)
    bubbles = detect_bubbles_fn(image)
    
    image_height = image.shape[0]
    
    sorted_panels, sorted_masks = sort_panels_manga_order(
        panel_bboxes, panel_masks, image_height, num_rows
    )
    
    ordered_bubbles = get_ordered_bubbles(
        bubbles, panel_bboxes, panel_masks, image_height, num_rows
    )
    
    return sorted_panels, sorted_masks, ordered_bubbles

def process_manga_page(
    image: np.ndarray,
    detect_panels_fn,
    detect_bubbles_fn,
    num_rows: int = 4
) -> Tuple[List[List[float]], List[np.ndarray], List[Dict[str, Any]]]:
    """
    Process manga page with double-page spread handling.
    Returns panels, masks, and bubbles in reading order.
    """
    if not is_double_page(image):
        return process_single_page(image, detect_panels_fn, detect_bubbles_fn, num_rows)
    
    height, full_width = image.shape[:2]
    mid_x = full_width // 2
    
    # Split into right and left pages
    right_page, left_page = split_double_page(image)
    
    # Process right page first (manga reading order)
    right_panels, right_masks, right_bubbles = process_single_page(
        right_page, detect_panels_fn, detect_bubbles_fn, num_rows
    )
    
    # Process left page
    left_panels, left_masks, left_bubbles = process_single_page(
        left_page, detect_panels_fn, detect_bubbles_fn, num_rows
    )
    
    # Adjust coordinates back to full image
    # Right page results (add mid_x offset since right half starts at mid)
    adjusted_right_panels = [
        [x1 + mid_x, y1, x2 + mid_x, y2] 
        for x1, y1, x2, y2 in right_panels
    ]
    adjusted_right_masks = [
        adjust_mask_for_right_page(mask, full_width) 
        for mask in right_masks
    ]
    adjusted_right_bubbles = [
        adjust_bubble_for_right_page(b, mid_x) 
        for b in right_bubbles
    ]
    
    # Left page results (coordinates stay as-is since left half starts at 0)
    adjusted_left_masks = [
        adjust_mask_for_left_page(mask, 0, full_width) 
        for mask in left_masks
    ]
    
    # Merge: right page first, then left page
    all_panels = adjusted_right_panels + left_panels
    all_masks = adjusted_right_masks + adjusted_left_masks
    all_bubbles = adjusted_right_bubbles + left_bubbles
    
    return all_panels, all_masks, all_bubbles

In [ ]:
import torch
import cv2
import numpy as np
from typing import List, Tuple

def yolo_masks_to_numpy(
    masks_tensor: torch.Tensor, 
    target_height: int, 
    target_width: int
) -> List[np.ndarray]:
    """Convert YOLO mask tensor to list of numpy masks resized to image dimensions"""
    if masks_tensor is None or len(masks_tensor) == 0:
        return []
    
    masks_np = masks_tensor.cpu().numpy()
    
    resized_masks = []
    for mask in masks_np:
        resized = cv2.resize(
            mask.astype(np.float32),
            (target_width, target_height),
            interpolation=cv2.INTER_LINEAR
        )
        resized_masks.append((resized > 0.5).astype(np.uint8))
    
    return resized_masks

def process_yolo_output(
    orig_img: np.ndarray,
    list_bboxes: List[List[float]],
    masks_tensor: torch.Tensor,
    probs: torch.Tensor,
    conf_threshold: float = 0.5
) -> Tuple[List[List[float]], List[np.ndarray], List[float]]:
    """Process raw YOLO output into compatible format"""
    height, width = orig_img.shape[:2]
    
    # Convert masks to numpy and resize
    masks = yolo_masks_to_numpy(masks_tensor, height, width)
    
    # Filter by confidence
    probs_np = probs.cpu().numpy()
    
    filtered_bboxes = []
    filtered_masks = []
    filtered_probs = []
    
    for bbox, mask, prob in zip(list_bboxes, masks, probs_np):
        if prob >= conf_threshold:
            filtered_bboxes.append(bbox)
            filtered_masks.append(mask)
            filtered_probs.append(float(prob))
    
    return filtered_bboxes, filtered_masks, filtered_probs

In [ ]:
def detect_with_yolo(
    yolo_model: YoloSeg, 
    image: np.ndarray,
    conf_threshold: float = 0.5
) -> Tuple[List[List[float]], List[np.ndarray]]:
    """Wrapper to get YOLO detections in compatible format"""
    orig_img, list_bboxes, masks_tensor, probs = yolo_model.predict(image)
    
    bboxes, masks, _ = process_yolo_output(
        orig_img, list_bboxes, masks_tensor, probs, conf_threshold
    )
    
    return bboxes, masks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
import matplotlib.colors as mcolors
from typing import List, Dict, Any, Optional

def plot_panels_and_bubbles(
    image: np.ndarray,
    panel_bboxes: List[List[float]],
    bubbles: List[Dict[str, Any]],
    panel_masks: Optional[List[np.ndarray]] = None,
    figsize: Tuple[int, int] = (16, 12),
    show_panel_ids: bool = True,
    show_bubble_ids: bool = True,
    panel_color: str = 'blue',
    bubble_color: str = 'red',
    mask_alpha: float = 0.3,
    title: str = "Manga Panel & Bubble Order"
):
    """
    Plot manga image with panels and bubbles, showing their reading order IDs.
    """
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    
    # Display image
    if image.max() > 1.0:
        img_display = image / 255.0
    else:
        img_display = image
    
    # Handle BGR to RGB if needed
    if len(img_display.shape) == 3 and img_display.shape[2] == 3:
        ax.imshow(img_display[:, :, ::-1] if img_display.shape[2] == 3 else img_display)
    else:
        ax.imshow(img_display, cmap='gray')
    
    # Plot panel masks if available
    if panel_masks is not None:
        colors = list(mcolors.TABLEAU_COLORS.values())
        h, w = image.shape[:2]
        overlay = np.zeros((h, w, 4))
        
        for i, mask in enumerate(panel_masks):
            color = mcolors.to_rgba(colors[i % len(colors)], alpha=mask_alpha)
            if mask.shape != (h, w):
                mask_resized = cv2.resize(mask.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
            else:
                mask_resized = mask
            overlay[mask_resized > 0] = color
        
        ax.imshow(overlay)
    
    # Plot panel bounding boxes
    for i, bbox in enumerate(panel_bboxes):
        x1, y1, x2, y2 = bbox
        width = x2 - x1
        height = y2 - y1
        
        rect = Rectangle(
            (x1, y1), width, height,
            linewidth=3,
            edgecolor=panel_color,
            facecolor='none',
            linestyle='-'
        )
        ax.add_patch(rect)
        
        if show_panel_ids:
            ax.text(
                x1 + 5, y1 + 30,
                f'P{i}',
                fontsize=16,
                fontweight='bold',
                color='white',
                bbox=dict(boxstyle='round', facecolor=panel_color, alpha=0.8)
            )
    
    # Plot bubble bounding boxes
    for i, bubble in enumerate(bubbles):
        x1 = bubble['xmin']
        y1 = bubble['ymin']
        x2 = bubble['xmax']
        y2 = bubble['ymax']
        width = x2 - x1
        height = y2 - y1
        
        rect = Rectangle(
            (x1, y1), width, height,
            linewidth=2,
            edgecolor=bubble_color,
            facecolor='none',
            linestyle='--'
        )
        ax.add_patch(rect)
        
        if show_bubble_ids:
            center_x = (x1 + x2) / 2
            center_y = (y1 + y2) / 2
            ax.text(
                center_x, center_y,
                f'{i}',
                fontsize=12,
                fontweight='bold',
                color='white',
                ha='center',
                va='center',
                bbox=dict(boxstyle='circle', facecolor=bubble_color, alpha=0.9)
            )
    
    ax.set_title(title, fontsize=14)
    ax.axis('off')
    
    # Add legend
    legend_elements = [
        Rectangle((0, 0), 1, 1, linewidth=3, edgecolor=panel_color, facecolor='none', label=f'Panels (P0-P{len(panel_bboxes)-1})'),
        Rectangle((0, 0), 1, 1, linewidth=2, edgecolor=bubble_color, facecolor='none', linestyle='--', label=f'Bubbles (0-{len(bubbles)-1})')
    ]
    ax.legend(handles=legend_elements, loc='upper left', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    return fig, ax


def plot_reading_order_arrows(
    image: np.ndarray,
    bubbles: List[Dict[str, Any]],
    figsize: Tuple[int, int] = (16, 12),
    arrow_color: str = 'lime',
    title: str = "Bubble Reading Order"
):
    """
    Plot bubbles with arrows showing reading order.
    """
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    
    if image.max() > 1.0:
        img_display = image / 255.0
    else:
        img_display = image
    
    ax.imshow(img_display[:, :, ::-1] if len(img_display.shape) == 3 else img_display)
    
    centers = []
    for i, bubble in enumerate(bubbles):
        x1, y1 = bubble['xmin'], bubble['ymin']
        x2, y2 = bubble['xmax'], bubble['ymax']
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        centers.append((cx, cy))
        
        # Draw bubble box
        rect = Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2,
            edgecolor='red',
            facecolor='none'
        )
        ax.add_patch(rect)
        
        # Draw ID
        ax.text(
            cx, cy, f'{i}',
            fontsize=14, fontweight='bold', color='white',
            ha='center', va='center',
            bbox=dict(boxstyle='circle', facecolor='red', alpha=0.9)
        )
    
    # Draw arrows between consecutive bubbles
    for i in range(len(centers) - 1):
        x1, y1 = centers[i]
        x2, y2 = centers[i + 1]
        
        ax.annotate(
            '',
            xy=(x2, y2),
            xytext=(x1, y1),
            arrowprops=dict(
                arrowstyle='->',
                color=arrow_color,
                lw=2,
                connectionstyle='arc3,rad=0.1'
            )
        )
    
    ax.set_title(title, fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    return fig, ax

In [ ]:
# Detection functions for process_manga_page
def detect_panels(image):
    return detect_with_yolo(panel_detector, image, conf_threshold=0.5)

def detect_bubbles(image):
    bboxes, masks = detect_with_yolo(bubble_detector, image, conf_threshold=0.5)
    # Convert to bubble dict format
    bubbles = [
        {'xmin': b[0], 'ymin': b[1], 'xmax': b[2], 'ymax': b[3]}
        for b in bboxes
    ]
    return bubbles

# Process manga page (handles double-page spreads)
panels, masks, ordered_bubbles = process_manga_page(
    image=image,
    detect_panels_fn=detect_panels,
    detect_bubbles_fn=detect_bubbles,
    num_rows=3
)

# Plot panels and bubbles with IDs
plot_panels_and_bubbles(
    image=image,
    panel_bboxes=panels,
    bubbles=ordered_bubbles,
    panel_masks=masks,
    show_panel_ids=True,
    show_bubble_ids=True
)

# Plot reading order with arrows
plot_reading_order_arrows(
    image=image,
    bubbles=ordered_bubbles
)



detect panel --> group theo từng row --> order panel theo từng row ---------------
                                                                                 |
                                                                                 v
                                detect bubble --> split bubble --> group bubble theo panel --> group bubble theo từng row trong từng panel --> order bubble 